# Thesis Figures — Inline Charts

Architecture comparison bar charts, heatmaps, abstention analysis, efficiency scatter, and per-comparison side-by-side figures.
Called from `mt_experiment_thesis_final` via `%run`. Some cells use `all_reps_df` from the parent; others are self-contained (load from Delta).

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

# ============================================================
# THESIS FIGURES — Bar charts comparing architectures (avg ± std)
# ============================================================
archs = ['SAS', 'SAS_RAG', 'MAS_RAG', 'DYNAMIC_FILTERED_MAS_RAG']
arch_labels = ['SAS', 'SAS+RAG', 'MAS+RAG', 'Dynamic\nFiltered']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

# Compute per-rep means → then mean±std across reps
metrics = {}
for arch in archs:
    sub = all_reps_df[all_reps_df['architecture']==arch]
    rep_means = []
    for rep in [1, 2, 3]:
        rs = sub[sub['repetition']==rep]
        if rs.empty:
            continue
        rep_means.append({
            'f1': rs['answer_f1'].mean(),
            'prec': rs['answer_precision'].mean(),
            'rec': rs['answer_recall'].mean(),
            'sql_rate': (rs['sql_execution_status']=='success').mean(),
            'hal': rs['numeric_hallucination_risk'].mean(),
            'lat': rs['latency_seconds'].mean(),
            'tok': rs['total_tokens'].mean()
        })
    rm = pd.DataFrame(rep_means)
    metrics[arch] = {k: (rm[k].mean(), rm[k].std()) for k in rm.columns}

# --- Figure 1: Quality Metrics ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Architecture Comparison (Mean ± Std, 3 Reps × 25 Questions)', fontsize=14, fontweight='bold')

chart_configs = [
    ('F1 Score ↑', 'f1', axes[0,0]),
    ('Precision ↑', 'prec', axes[0,1]),
    ('Recall ↑', 'rec', axes[0,2]),
    ('SQL Success Rate ↑', 'sql_rate', axes[1,0]),
    ('Hallucination Risk ↓', 'hal', axes[1,1]),
    ('Latency (seconds) ↓', 'lat', axes[1,2]),
]

for title, metric, ax in chart_configs:
    means = [metrics[a][metric][0] for a in archs]
    stds  = [metrics[a][metric][1] for a in archs]
    bars = ax.bar(arch_labels, means, yerr=stds, capsize=5, color=colors,
                  alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(bottom=0)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{m:.2f}', ha='center', va='bottom', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# --- Figure 2: Token Efficiency ---
fig2, ax2 = plt.subplots(figsize=(10, 5))
tok_means = [metrics[a]['tok'][0] for a in archs]
tok_stds  = [metrics[a]['tok'][1] for a in archs]
bars = ax2.bar(arch_labels, tok_means, yerr=tok_stds, capsize=5, color=colors,
               alpha=0.85, edgecolor='black', linewidth=0.5)
ax2.set_title('Average Token Usage per Question (Mean ± Std)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Tokens')
for bar, m in zip(bars, tok_means):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 100,
            f'{m:,.0f}', ha='center', va='bottom', fontsize=10)
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# --- Figure 3: Per-Rep F1 Trend (stability visualization) ---
fig3, ax3 = plt.subplots(figsize=(10, 5))
for i, (arch, label) in enumerate(zip(archs, ['SAS', 'SAS+RAG', 'MAS+RAG', 'Dynamic Filtered'])):
    sub = all_reps_df[all_reps_df['architecture']==arch]
    rep_f1 = [sub[sub['repetition']==r]['answer_f1'].mean() for r in [1,2,3]]
    ax3.plot([1, 2, 3], rep_f1, 'o-', color=colors[i], label=label, linewidth=2, markersize=8)

ax3.set_xlabel('Repetition', fontsize=11)
ax3.set_ylabel('Mean F1 Score', fontsize=11)
ax3.set_title('F1 Score Stability Across 3 Repetitions', fontsize=12, fontweight='bold')
ax3.set_xticks([1, 2, 3])
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n\u2713 All architecture comparison figures generated")

In [0]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib
from matplotlib.patches import FancyBboxPatch
matplotlib.rcParams['figure.dpi'] = 150

# ── Load from Delta (self-contained cell) ──
df = spark.table("dev_forge_default.mt_davide.experiment_thesis_results").toPandas()
for c in ['answer_f1','answer_precision','answer_recall','numeric_hallucination_risk',
          'latency_seconds','total_tokens','claim_groundedness']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
df['abstained'] = df['abstained'].astype(str).isin(['True','true','1'])
df['answerable'] = df['answerable'].astype(bool)

archs = ['SAS','SAS_RAG','MAS_RAG','DYNAMIC_FILTERED_MAS_RAG']
labels = ['SAS','SAS+RAG','MAS+RAG','Dynamic']
colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0']
diffs = ['easy','medium','hard','very_hard']

# ================================================================
# FIG 1 — HEATMAP: F1 by Architecture × Difficulty
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

f1_matrix = np.zeros((len(archs), len(diffs)))
for i, a in enumerate(archs):
    for j, d in enumerate(diffs):
        sub = df[(df['architecture']==a) & (df['difficulty']==d)]
        f1_matrix[i,j] = sub['answer_f1'].mean() if not sub.empty else 0

im = axes[0].imshow(f1_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.5)
axes[0].set_xticks(range(len(diffs))); axes[0].set_xticklabels([d.replace('_',' ').title() for d in diffs], fontsize=10)
axes[0].set_yticks(range(len(labels))); axes[0].set_yticklabels(labels, fontsize=11)
for i in range(len(archs)):
    for j in range(len(diffs)):
        axes[0].text(j, i, f'{f1_matrix[i,j]:.3f}', ha='center', va='center',
                     fontsize=11, fontweight='bold', color='white' if f1_matrix[i,j] > 0.25 else 'black')
axes[0].set_title('F1 Score by Architecture × Difficulty', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0], shrink=0.8, label='F1')

gr_matrix = np.zeros((len(archs), len(diffs)))
for i, a in enumerate(archs):
    for j, d in enumerate(diffs):
        sub = df[(df['architecture']==a) & (df['difficulty']==d)]
        g = sub['claim_groundedness'].dropna()
        gr_matrix[i,j] = g.mean() if len(g) > 0 else 0

im2 = axes[1].imshow(gr_matrix, cmap='YlGn', aspect='auto', vmin=0, vmax=1.0)
axes[1].set_xticks(range(len(diffs))); axes[1].set_xticklabels([d.replace('_',' ').title() for d in diffs], fontsize=10)
axes[1].set_yticks(range(len(labels))); axes[1].set_yticklabels(labels, fontsize=11)
for i in range(len(archs)):
    for j in range(len(diffs)):
        axes[1].text(j, i, f'{gr_matrix[i,j]:.2f}', ha='center', va='center',
                     fontsize=11, fontweight='bold', color='white' if gr_matrix[i,j] > 0.55 else 'black')
axes[1].set_title('Groundedness by Architecture × Difficulty', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Groundedness')
plt.tight_layout()
plt.show()

# ================================================================
# FIG 2 — ABSTENTION BEHAVIOR
# ================================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5.5))
bar_data = []
for a in archs:
    sub = df[df['architecture']==a]
    unans = sub[sub['answerable']==False]
    ans = sub[sub['answerable']==True]
    correct_abstain = unans['abstained'].sum()
    missed_unans = len(unans) - correct_abstain
    false_abstain = ans['abstained'].sum()
    answered = len(ans) - false_abstain
    bar_data.append({'arch': a, 'Correct Abstention': correct_abstain,
                     'Missed (should abstain)': missed_unans,
                     'False Abstention': false_abstain, 'Answered': answered})
bdf = pd.DataFrame(bar_data)
x = np.arange(len(labels)); w = 0.55; bottom = np.zeros(len(labels))
cat_colors = ['#66BB6A','#EF5350','#FFA726','#42A5F5']
for cat, col in zip(['Correct Abstention','Missed (should abstain)','False Abstention','Answered'], cat_colors):
    vals = bdf[cat].values
    axes2[0].bar(x, vals, w, bottom=bottom, label=cat, color=col, edgecolor='white', linewidth=0.5)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 2:
            axes2[0].text(i, b + v/2, str(int(v)), ha='center', va='center', fontsize=9, fontweight='bold')
    bottom += vals
axes2[0].set_xticks(x); axes2[0].set_xticklabels(labels, fontsize=11)
axes2[0].set_ylabel('Number of Runs (out of 75)', fontsize=10)
axes2[0].set_title('Abstention Behavior Breakdown', fontsize=12, fontweight='bold')
axes2[0].legend(loc='upper left', fontsize=9, framealpha=0.9); axes2[0].set_ylim(0, 82)

fab_rates = []
for a in archs:
    sub = df[df['architecture']==a]
    pff = sub['post_failure_fabrication'].astype(str).isin(['True','true','1']).sum() if 'post_failure_fabrication' in sub.columns else 0
    fab_rates.append(pff / len(sub) * 100)
bars = axes2[1].bar(labels, fab_rates, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5, width=0.55)
for bar, v in zip(bars, fab_rates):
    axes2[1].text(bar.get_x()+bar.get_width()/2., bar.get_height()+1, f'{v:.0f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes2[1].set_ylabel('Fabrication Rate (%)'); axes2[1].set_title('Post-Failure Fabrication Rate', fontsize=12, fontweight='bold')
axes2[1].set_ylim(0, 70); axes2[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

# ================================================================
# FIG 3 — SQL SUCCESS BY DIFFICULTY
# ================================================================
fig3, ax3 = plt.subplots(figsize=(12, 5.5)); bar_w = 0.18; x3 = np.arange(len(diffs))
for i, (a, lbl, col) in enumerate(zip(archs, labels, colors)):
    rates = [(df[(df['architecture']==a)&(df['difficulty']==d)]['sql_execution_status']=='success').mean()*100 if not df[(df['architecture']==a)&(df['difficulty']==d)].empty else 0 for d in diffs]
    bars = ax3.bar(x3 + i*bar_w, rates, bar_w, label=lbl, color=col, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, rates):
        if v > 0: ax3.text(bar.get_x()+bar.get_width()/2., bar.get_height()+1, f'{v:.0f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax3.set_xticks(x3 + 1.5*bar_w); ax3.set_xticklabels([d.replace('_',' ').title() for d in diffs], fontsize=11)
ax3.set_ylabel('SQL Success Rate (%)', fontsize=11); ax3.set_title('SQL Generation Success by Difficulty Level', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10); ax3.set_ylim(0, 115); ax3.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

# ================================================================
# FIG 4 — EFFICIENCY SCATTER: F1 vs Tokens
# ================================================================
fig4, ax4 = plt.subplots(figsize=(10, 7))
for a, lbl, col in zip(archs, labels, colors):
    sub = df[df['architecture']==a]
    f1_mean = sub['answer_f1'].mean(); tok_mean = sub['total_tokens'].mean()
    lat_mean = sub['latency_seconds'].mean(); grnd_mean = sub['claim_groundedness'].dropna().mean()
    ax4.scatter(tok_mean, f1_mean, s=lat_mean*40, c=col, alpha=0.7, edgecolor='black', linewidth=1.5, zorder=5)
    ax4.annotate(f'{lbl}\nF1={f1_mean:.3f}\nGrnd={grnd_mean:.2f}\n{lat_mean:.1f}s',
                 (tok_mean, f1_mean), textcoords='offset points', xytext=(15, 10 if a != 'SAS' else -25),
                 fontsize=9, arrowprops=dict(arrowstyle='->', color='gray', lw=0.8),
                 bbox=dict(boxstyle='round,pad=0.3', facecolor=col, alpha=0.15))
ax4.annotate('', xy=(8000, 0.28), xytext=(32000, 0.05), arrowprops=dict(arrowstyle='->', color='green', lw=2, linestyle='--'))
ax4.text(15000, 0.18, 'Better\n(↑ Quality, ↓ Cost)', fontsize=9, color='green', fontstyle='italic', ha='center')
ax4.set_xlabel('Average Tokens per Question', fontsize=11); ax4.set_ylabel('Average F1 Score', fontsize=11)
ax4.set_title('Quality vs. Cost Trade-off (bubble size = latency)', fontsize=12, fontweight='bold')
ax4.grid(alpha=0.3); plt.tight_layout(); plt.show()

# ================================================================
# FIG 5 — PAIRWISE DELTA CHART
# ================================================================
fig5, axes5 = plt.subplots(1, 3, figsize=(18, 5.5))
pairs = [('SAS','SAS_RAG','SAS → SAS+RAG\n(Effect of RAG)','SAS','SAS+RAG'),
         ('SAS_RAG','MAS_RAG','SAS+RAG → MAS+RAG\n(Effect of Multi-Agent)','SAS+RAG','MAS+RAG'),
         ('MAS_RAG','DYNAMIC_FILTERED_MAS_RAG','MAS+RAG → Dynamic\n(Effect of Filtering)','MAS+RAG','Dynamic')]
metric_names = ['F1','Precision','Recall','Groundedness','SQL Rate']
metric_cols = ['answer_f1','answer_precision','answer_recall','claim_groundedness', None]
for ax, (a1,a2,title,l1,l2) in zip(axes5, pairs):
    deltas = []
    for mname, mcol in zip(metric_names, metric_cols):
        if mcol: v1 = df[df['architecture']==a1][mcol].dropna().mean(); v2 = df[df['architecture']==a2][mcol].dropna().mean()
        else: v1 = (df[df['architecture']==a1]['sql_execution_status']=='success').mean(); v2 = (df[df['architecture']==a2]['sql_execution_status']=='success').mean()
        deltas.append(((v2 - v1) / v1 * 100) if v1 > 0 else 0)
    x5 = np.arange(len(metric_names))
    bar_colors = ['#66BB6A' if d > 0 else '#EF5350' for d in deltas]
    bars = ax.barh(x5, deltas, color=bar_colors, alpha=0.8, edgecolor='black', linewidth=0.5, height=0.5)
    for bar, d in zip(bars, deltas):
        offset = 3 if d >= 0 else -3; ha = 'left' if d >= 0 else 'right'
        ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height()/2, f'{d:+.1f}%', ha=ha, va='center', fontsize=10, fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8); ax.set_yticks(x5); ax.set_yticklabels(metric_names, fontsize=10)
    ax.set_xlabel('Change (%)', fontsize=10); ax.set_title(title, fontsize=11, fontweight='bold'); ax.grid(axis='x', alpha=0.3)
    max_abs = max(abs(min(deltas)), abs(max(deltas)), 50); ax.set_xlim(-max_abs*1.3, max_abs*1.3)
plt.tight_layout(); plt.show()

print("\n\u2713 All detailed comparison figures generated")

In [0]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams['figure.dpi'] = 150

df = spark.table("dev_forge_default.mt_davide.experiment_thesis_results").toPandas()
for c in ['answer_f1','answer_precision','answer_recall','numeric_hallucination_risk',
          'latency_seconds','total_tokens','claim_groundedness']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['abstained'] = df['abstained'].astype(str).isin(['True','true','1'])
df['answerable'] = df['answerable'].astype(bool)

diffs = ['easy','medium','hard','very_hard']
diff_labels = ['Easy','Medium','Hard','Very Hard']

# ================================================================
# FIGURE A — SAS vs SAS+RAG
# ================================================================
fig_a, (ax_a1, ax_a2) = plt.subplots(1, 2, figsize=(16, 5.5))
w = 0.35; x = np.arange(len(diffs))
for i, (arch, lbl, col) in enumerate([('SAS','SAS','#2196F3'), ('SAS_RAG','SAS+RAG','#4CAF50')]):
    vals = [df[(df['architecture']==arch) & (df['difficulty']==d)]['answer_f1'].mean() for d in diffs]
    bars = ax_a1.bar(x + i*w, vals, w, label=lbl, color=col, alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, vals): ax_a1.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_a1.set_xticks(x + w/2); ax_a1.set_xticklabels(diff_labels, fontsize=11)
ax_a1.set_ylabel('F1 Score', fontsize=11); ax_a1.set_title('F1 by Difficulty: SAS vs SAS+RAG', fontsize=12, fontweight='bold')
ax_a1.legend(fontsize=10); ax_a1.set_ylim(0, 0.32); ax_a1.grid(axis='y', alpha=0.3)

metric_labels = ['F1', 'Precision', 'Recall', 'Groundedness', 'SQL Rate', '1 - Halluc.']
metric_cols   = ['answer_f1','answer_precision','answer_recall','claim_groundedness', None, None]
sas_vals, rag_vals = [], []
for ml, mc in zip(metric_labels, metric_cols):
    s = df[df['architecture']=='SAS']; r = df[df['architecture']=='SAS_RAG']
    if ml == 'SQL Rate': sas_vals.append((s['sql_execution_status']=='success').mean()); rag_vals.append((r['sql_execution_status']=='success').mean())
    elif ml == '1 - Halluc.': sas_vals.append(1 - s['numeric_hallucination_risk'].mean()); rag_vals.append(1 - r['numeric_hallucination_risk'].mean())
    else: sas_vals.append(s[mc].dropna().mean()); rag_vals.append(r[mc].dropna().mean())
y = np.arange(len(metric_labels))
ax_a2.barh(y + 0.18, sas_vals, 0.32, label='SAS', color='#2196F3', alpha=0.85, edgecolor='black', linewidth=0.5)
ax_a2.barh(y - 0.18, rag_vals, 0.32, label='SAS+RAG', color='#4CAF50', alpha=0.85, edgecolor='black', linewidth=0.5)
for yi, (sv, rv) in enumerate(zip(sas_vals, rag_vals)):
    ax_a2.text(max(sv, rv) + 0.02, yi, f'\u0394={((rv-sv)/sv*100):+.0f}%' if sv > 0 else '', fontsize=9, va='center', color='#EF5350' if rv < sv else '#66BB6A', fontweight='bold')
ax_a2.set_yticks(y); ax_a2.set_yticklabels(metric_labels, fontsize=10); ax_a2.set_xlabel('Score'); ax_a2.set_xlim(0, 0.7)
ax_a2.set_title('SAS vs SAS+RAG: All Quality Metrics', fontsize=12, fontweight='bold'); ax_a2.legend(fontsize=10, loc='lower right'); ax_a2.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

# ================================================================
# FIGURE B — SAS+RAG vs MAS+RAG
# ================================================================
fig_b, (ax_b1, ax_b2) = plt.subplots(1, 2, figsize=(16, 5.5))
w = 0.35; x = np.arange(len(diffs))
for i, (arch, lbl, col) in enumerate([('SAS_RAG','SAS+RAG','#4CAF50'), ('MAS_RAG','MAS+RAG','#FF9800')]):
    vals = [df[(df['architecture']==arch) & (df['difficulty']==d)]['answer_f1'].mean() for d in diffs]
    bars = ax_b1.bar(x + i*w, vals, w, label=lbl, color=col, alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, vals): ax_b1.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_b1.set_xticks(x + w/2); ax_b1.set_xticklabels(diff_labels, fontsize=11); ax_b1.set_ylabel('F1 Score', fontsize=11)
ax_b1.set_title('F1 by Difficulty: SAS+RAG vs MAS+RAG', fontsize=12, fontweight='bold'); ax_b1.legend(fontsize=10); ax_b1.set_ylim(0, 0.45); ax_b1.grid(axis='y', alpha=0.3)

metric_labels2 = ['F1','Precision','Recall','Groundedness','SQL Rate','1 - Halluc.','Abstain (unans.)']
rag_v2, mas_v2 = [], []
for ml in metric_labels2:
    r = df[df['architecture']=='SAS_RAG']; m = df[df['architecture']=='MAS_RAG']
    if ml == 'SQL Rate': rag_v2.append((r['sql_execution_status']=='success').mean()); mas_v2.append((m['sql_execution_status']=='success').mean())
    elif ml == '1 - Halluc.': rag_v2.append(1 - r['numeric_hallucination_risk'].mean()); mas_v2.append(1 - m['numeric_hallucination_risk'].mean())
    elif ml == 'Abstain (unans.)': ru = r[r['answerable']==False]; mu = m[m['answerable']==False]; rag_v2.append(ru['abstained'].mean()); mas_v2.append(mu['abstained'].mean())
    elif ml == 'F1': rag_v2.append(r['answer_f1'].mean()); mas_v2.append(m['answer_f1'].mean())
    elif ml == 'Precision': rag_v2.append(r['answer_precision'].mean()); mas_v2.append(m['answer_precision'].mean())
    elif ml == 'Recall': rag_v2.append(r['answer_recall'].mean()); mas_v2.append(m['answer_recall'].mean())
    elif ml == 'Groundedness': rag_v2.append(r['claim_groundedness'].dropna().mean()); mas_v2.append(m['claim_groundedness'].dropna().mean())
y2 = np.arange(len(metric_labels2))
ax_b2.barh(y2 + 0.18, rag_v2, 0.32, label='SAS+RAG', color='#4CAF50', alpha=0.85, edgecolor='black', linewidth=0.5)
ax_b2.barh(y2 - 0.18, mas_v2, 0.32, label='MAS+RAG', color='#FF9800', alpha=0.85, edgecolor='black', linewidth=0.5)
for yi, (rv, mv) in enumerate(zip(rag_v2, mas_v2)):
    delta_pct = ((mv-rv)/rv*100) if rv > 0 else float('inf')
    ax_b2.text(max(rv, mv) + 0.02, yi, f'\u0394={delta_pct:+.0f}%' if abs(delta_pct) < 1000 else '\u221e', fontsize=9, va='center', color='#66BB6A' if mv > rv else '#EF5350', fontweight='bold')
ax_b2.set_yticks(y2); ax_b2.set_yticklabels(metric_labels2, fontsize=10); ax_b2.set_xlabel('Score'); ax_b2.set_xlim(0, 0.7)
ax_b2.set_title('SAS+RAG vs MAS+RAG: All Quality Metrics', fontsize=12, fontweight='bold'); ax_b2.legend(fontsize=10, loc='lower right'); ax_b2.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

# ================================================================
# FIGURE C — MAS+RAG vs DYNAMIC
# ================================================================
fig_c, (ax_c1, ax_c2) = plt.subplots(1, 2, figsize=(16, 5.5))
w = 0.35; x = np.arange(len(diffs))
for i, (arch, lbl, col) in enumerate([('MAS_RAG','MAS+RAG','#FF9800'), ('DYNAMIC_FILTERED_MAS_RAG','Dynamic','#9C27B0')]):
    vals = [df[(df['architecture']==arch) & (df['difficulty']==d)]['answer_f1'].mean() for d in diffs]
    bars = ax_c1.bar(x + i*w, vals, w, label=lbl, color=col, alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, vals): ax_c1.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_c1.set_xticks(x + w/2); ax_c1.set_xticklabels(diff_labels, fontsize=11); ax_c1.set_ylabel('F1 Score', fontsize=11)
ax_c1.set_title('F1 by Difficulty: MAS+RAG vs Dynamic', fontsize=12, fontweight='bold'); ax_c1.legend(fontsize=10); ax_c1.set_ylim(0, 0.52); ax_c1.grid(axis='y', alpha=0.3)

metric_labels3 = ['F1','Precision','Recall','Groundedness','SQL Rate','1 - Halluc.','Abstain (unans.)']
mas_v3, dyn_v3 = [], []
for ml in metric_labels3:
    m = df[df['architecture']=='MAS_RAG']; d = df[df['architecture']=='DYNAMIC_FILTERED_MAS_RAG']
    if ml == 'SQL Rate': mas_v3.append((m['sql_execution_status']=='success').mean()); dyn_v3.append((d['sql_execution_status']=='success').mean())
    elif ml == '1 - Halluc.': mas_v3.append(1 - m['numeric_hallucination_risk'].mean()); dyn_v3.append(1 - d['numeric_hallucination_risk'].mean())
    elif ml == 'Abstain (unans.)': mu = m[m['answerable']==False]; du = d[d['answerable']==False]; mas_v3.append(mu['abstained'].mean()); dyn_v3.append(du['abstained'].mean())
    elif ml == 'F1': mas_v3.append(m['answer_f1'].mean()); dyn_v3.append(d['answer_f1'].mean())
    elif ml == 'Precision': mas_v3.append(m['answer_precision'].mean()); dyn_v3.append(d['answer_precision'].mean())
    elif ml == 'Recall': mas_v3.append(m['answer_recall'].mean()); dyn_v3.append(d['answer_recall'].mean())
    elif ml == 'Groundedness': mas_v3.append(m['claim_groundedness'].dropna().mean()); dyn_v3.append(d['claim_groundedness'].dropna().mean())
y3 = np.arange(len(metric_labels3))
ax_c2.barh(y3 + 0.18, mas_v3, 0.32, label='MAS+RAG', color='#FF9800', alpha=0.85, edgecolor='black', linewidth=0.5)
ax_c2.barh(y3 - 0.18, dyn_v3, 0.32, label='Dynamic', color='#9C27B0', alpha=0.85, edgecolor='black', linewidth=0.5)
for yi, (mv, dv) in enumerate(zip(mas_v3, dyn_v3)):
    delta_pct = ((dv-mv)/mv*100) if mv > 0 else float('inf')
    ax_c2.text(max(mv, dv) + 0.02, yi, f'\u0394={delta_pct:+.0f}%' if abs(delta_pct) < 1000 else '\u221e', fontsize=9, va='center', color='#66BB6A' if dv > mv else '#EF5350', fontweight='bold')
ax_c2.set_yticks(y3); ax_c2.set_yticklabels(metric_labels3, fontsize=10); ax_c2.set_xlabel('Score'); ax_c2.set_xlim(0, 0.85)
ax_c2.set_title('MAS+RAG vs Dynamic: All Quality Metrics', fontsize=12, fontweight='bold'); ax_c2.legend(fontsize=10, loc='lower right'); ax_c2.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print('\n\u2713 Per-comparison figures generated (A: SAS vs SAS+RAG, B: SAS+RAG vs MAS+RAG, C: MAS+RAG vs Dynamic)')